In [2]:
from pathlib import Path
import pandas as pd
from data_ingestion import load_all_raw
from feature_engineering import build_subject_snapshot
from scoring import compute_clean_patient_flags, compute_dqi

raw = load_all_raw()
subject_df = build_subject_snapshot(raw)
subject_df = compute_clean_patient_flags(subject_df)
subject_df = compute_dqi(subject_df)

# # --- Normalize identifier columns for Parquet ---
# id_cols = ["project_name", "site_id", "subject_id"]
#
# for c in id_cols:
#     if c in subject_df.columns:
#         subject_df[c] = subject_df[c].astype(str)

def make_parquet_safe(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].astype(str)
    return out
subject_df = make_parquet_safe(subject_df)
subject_df.to_parquet("C:/Users/Aritri Baidya/Desktop/Novartis/subject_site_snapshot.parquet", index=False)

# Summary stats
subject_df["dqi"].describe()
subject_df["dqi_band"].value_counts(normalize=True)

# Site-level aggregation
site_df = subject_df.groupby(["project_name", "site_id"], as_index=False).agg(
    mean_dqi=("dqi", "mean"),
    pct_clean=("clean_patient", "mean"),
    n_subjects=("subject_id", "nunique"),
    n_red=("dqi_band", lambda x: (x=="Red").sum())
)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

features = [
    "n_missing_visits", "n_missing_pages", "n_open_queries",
    "n_nonconformant_pages", "n_lab_issues", "n_uncoded_terms",
    "n_open_edrr_issues", "n_sae_pending_actions",
    "pct_crfs_verified", "pct_crfs_signed", "pct_crfs_overdue"
]

X = subject_df[features].fillna(0)
y = subject_df["clean_patient"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced"
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        21

    accuracy                           1.00        21
   macro avg       1.00      1.00      1.00        21
weighted avg       1.00      1.00      1.00        21



In [ ]:
%load_ext autoreload
%autoreload 2

from data_ingestion import DATA_DIR, find_file, load_all_raw

print("DATA_DIR:", DATA_DIR)
for f in DATA_DIR.glob("*.xls*"):
    print(f.name)

raw = load_all_raw()


print("Visit Projection Tracker columns:")
for c in raw["visits"].columns:
    print(repr(c))

print("Missing Pages columns:")
for c in raw["missing_pages"].columns:
    print(repr(c))

print("Missing Lab columns:")
for c in raw["missing_lab"].columns:
    print(repr(c))

